In [ ]:
import os
import glob
import cv2 as cv
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from IPython.display import display, Image as IPyImage

# 导入你自定义的模型文件 (确保这些文件在同级目录)
from Model import DeePixBiS
# from Loss import PixWiseBCELoss
# from Metrics import predict, test_accuracy, test_loss

# 1. 设备配置 (自动使用 GPU 加速)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前使用的计算设备: {device}")

# 2. 加载模型
model = DeePixBiS().to(device)
model.load_state_dict(torch.load('./DeePixBiS.pth', map_location=device))
model.eval()
print("DeePixBiS 活体检测模型加载成功！")

# 3. 图像预处理配置
tfms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. 加载人脸检测器 (如果报错找不到路径，可换成 cv.data.haarcascades + 'haarcascade_frontalface_default.xml')
faceClassifier = cv.CascadeClassifier('Classifiers/haarface.xml')



In [ ]:

def process_single_frame(img_bgr):
    """
    输入一帧 BGR 图像，检测人脸、进行活体判断，并绘制结果框
    """
    # 拷贝图像防止修改原始数据
    img_draw = img_bgr.copy()
    grey = cv.cvtColor(img_draw, cv.COLOR_BGR2GRAY)
    faces = faceClassifier.detectMultiScale(grey, scaleFactor=1.1, minNeighbors=4)

    for x, y, w, h in faces:
        # 提取人脸区域并转为 RGB
        faceRegion = img_bgr[y:y + h, x:x + w]
        faceRegion_rgb = cv.cvtColor(faceRegion, cv.COLOR_BGR2RGB)

        # 预处理并送入模型
        tensor_face = tfms(faceRegion_rgb).unsqueeze(0).to(device)

        with torch.no_grad():
            mask, binary = model.forward(tensor_face)
            res = torch.mean(mask).item() # 获取活体分数

        # 根据阈值判断真假并绘制
        color = (0, 0, 255) if res < 0.5 else (0, 255, 0) # 假脸红色，真脸绿色
        label = f'Fake: {res:.2f}' if res < 0.5 else f'Real: {res:.2f}'
        
        cv.rectangle(img_draw, (x, y), (x + w, y + h), color, 2)
        cv.putText(img_draw, label, (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    return img_draw

def show_image_in_jupyter(img_bgr, title="Image"):
    """在 Jupyter 中展示静态图片 (Matplotlib方式)"""
    img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 6))
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()



In [ ]:

def run_on_image(image_path):
    """处理单张图片"""
    if not os.path.exists(image_path):
        print(f"图片不存在: {image_path}")
        return
    img = cv.imread(image_path)
    result_img = process_single_frame(img)
    show_image_in_jupyter(result_img, title=f"Result: {os.path.basename(image_path)}")

def run_on_folder(folder_path):
    """处理文件夹下所有图片"""
    if not os.path.exists(folder_path):
        print(f"文件夹不存在: {folder_path}")
        return
    
    # 支持常见的图片格式
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
        
    print(f"在 {folder_path} 中找到 {len(image_paths)} 张图片。")
    for img_path in image_paths:
        run_on_image(img_path)

def run_on_video(video_source):
    """
    处理视频文件或摄像头，并在 Jupyter 中丝滑渲染
    video_source: 视频文件路径 (字符串) 或 摄像头索引 (通常为 0)
    """
    camera = cv.VideoCapture(video_source)
    if not camera.isOpened():
        print(f"无法打开视频源: {video_source}")
        return

    try:
        # 创建一个 Jupyter 原地更新的显示句柄 (解决 clear_output 闪烁问题)
        display_handle = display(None, display_id=True)
        
        print("开始处理视频流... (在终端按 '停止/Interrupt' 按钮结束)")
        
        while True:
            ret, frame = camera.read()
            if not ret:
                print("视频流结束。")
                break
                
            # 处理当前帧
            result_frame = process_single_frame(frame)
            
            # 将处理后的 BGR 帧编码为 JPEG 格式字节流，送给 IPython 展示
            _, encoded_img = cv.imencode('.jpg', result_frame)
            display_handle.update(IPyImage(data=encoded_img.tobytes()))
            
    except KeyboardInterrupt:
        # 捕获手动停止中断
        print("手动停止视频处理。")
    finally:
        camera.release()
        print("摄像头/视频已释放。")



In [ ]:

# --- 模式 1: 测试单张图片 ---
# run_on_image('./test_images/sample1.jpg')

# --- 模式 2: 测试整个文件夹 ---
# run_on_folder('./test_images/')

# --- 模式 3: 测试视频文件 ---
video_path = '/supercloud/llm-code/scc/scc/Liveness_Detection/炫彩闪烁活体/20230828 头模 正常光/FaceCollect/1693184996070_note_1_1_toumo_6.avi'
run_on_video(video_path)

# --- 模式 4: 测试本地摄像头 (如果服务器未连接摄像头，请勿运行此项) ---
# run_on_video(0)